# Chicago Violations Data Pipeline

## Overview

A medallion-architecture pipeline (bronze → silver → gold) for Chicago
ordinance violations data on AWS Glue Interactive Sessions, S3 Tables /
Apache Iceberg, and Amazon Deequ.

```
S3 raw CSV → Bronze (DataFrame) → Silver (Iceberg, partitioned by ward)
            → Deequ validation → Gold (Iceberg) + Quarantine (Iceberg)
```

### Stages
1. **Bronze** — read CSV from S3, infer schema, preserve every column.
2. **Silver** — normalise column names, parse dates explicitly, strip
   thousands separators on currency, cast to typed columns (Decimal for
   money, Double for coords, Integer for ward), append to a partitioned
   Iceberg table.
3. **Deequ validation** — completeness / uniqueness / range checks.
4. **Gold** — append validated rows to the gold Iceberg table; failed
   rows are written to a quarantine table for investigation.

### Stack
- AWS Glue Interactive Sessions (Spark 3.3+)
- S3 Tables + Apache Iceberg via the AWS Glue Data Catalog
- PyDeequ for data-quality validation


## Glue session configuration

Use the Glue interactive-session magics to size compute and pull in the
Iceberg + Deequ runtime jars. Update the JAR S3 paths to point at copies
in your account.


In [ ]:
%glue_version 4.0


In [ ]:
%idle_timeout 30


In [ ]:
%worker_type G.1X
%number_of_workers 2


In [ ]:
# Substitute your account / region. The job role needs s3:GetObject on these.
%extra_jars s3://aws-glue-assets-<account>-<region>/jars/deequ-2.0.3-spark-3.3.jar,s3://aws-glue-assets-<account>-<region>/jars/iceberg-spark-runtime-3.3_2.12-1.4.3.jar


In [ ]:
%%configure
{
    "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "--datalake-formats": "iceberg"
}


## Pipeline configuration

All knobs live here. The bronze source is on S3 because Glue interactive
workers cannot read the local notebook filesystem.


In [ ]:
# S3 source — upload the sample CSV to this prefix before running.
SOURCE_CSV_PATH = "s3://chicago-violations-raw-us-east-1-123456789012/raw/chicago-ordinance-violations-sample.csv"

# S3 Tables bucket holding the lakehouse.
S3_TABLES_BUCKET_ARN = "arn:aws:s3tables:us-east-1:123456789012:bucket/chicago-violations-lakehouse"

CATALOG_NAME      = "chicago_lakehouse"
NAMESPACE         = "violations"
SILVER_TABLE      = f"{CATALOG_NAME}.{NAMESPACE}.violations_silver"
GOLD_TABLE        = f"{CATALOG_NAME}.{NAMESPACE}.violations_gold"
QUARANTINE_TABLE  = f"{CATALOG_NAME}.{NAMESPACE}.violations_quarantine"
PARTITION_COLUMN  = "ward"

# Source-specific date format.
DATE_FORMAT_LONG  = "M/d/yy H:mm"   # last_modified_date, violation_date, hearing_date with time
DATE_FORMAT_SHORT = "M/d/yy"        # hearing_date when no time component

for k, v in dict(SOURCE_CSV_PATH=SOURCE_CSV_PATH, S3_TABLES_BUCKET_ARN=S3_TABLES_BUCKET_ARN,
                 SILVER_TABLE=SILVER_TABLE, GOLD_TABLE=GOLD_TABLE,
                 QUARANTINE_TABLE=QUARANTINE_TABLE, PARTITION_COLUMN=PARTITION_COLUMN).items():
    print(f"{k}: {v}")


## Iceberg catalog configuration

In [ ]:
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", S3_TABLES_BUCKET_ARN)
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG_NAME}.{NAMESPACE}")

for prop in ("catalog-impl", "warehouse", "io-impl"):
    print(prop, "->", spark.conf.get(f"spark.sql.catalog.{CATALOG_NAME}.{prop}"))


### Required IAM (Glue execution role)
- `s3tables:Get/Create/Update/Delete/PutTableData/GetTableData` on the lakehouse bucket
- `glue:Get/Create/Update/Delete Database|Table` on the catalog
- `s3:GetObject` on the raw bucket and the JAR locations
- `kms:Decrypt`/`kms:GenerateDataKey` on any CMK protecting the data


## Imports & timing helpers

In [ ]:
import time
from contextlib import contextmanager

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, DoubleType, IntegerType

from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationResult, VerificationSuite

stage_times = {}

@contextmanager
def stage(name):
    print(f"\n=== {name} ===")
    start = time.time()
    try:
        yield
    finally:
        elapsed = time.time() - start
        stage_times[name] = elapsed
        print(f"=== {name} done in {elapsed:.2f}s ===")


## Bronze stage — read raw CSV from S3

In [ ]:
with stage("bronze"):
    df_bronze = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .option("escape", '"')
             .csv(SOURCE_CSV_PATH)
    )
    bronze_count = df_bronze.count()
    print(f"rows: {bronze_count:,}  cols: {len(df_bronze.columns)}")
    df_bronze.printSchema()
    df_bronze.show(5, truncate=False)


## Silver stage — typed, partitioned Iceberg table

Cleanups applied here:
- column names → `lower_snake_case`
- currency strings like `"2,000"` → `DecimalType(10, 2)`
- explicit `to_timestamp` with the source's `M/d/yy H:mm` format
- coordinates as `DoubleType`, `ward` / `street_number` as `IntegerType`
- audit columns: `_ingested_at`, `_source_path`
- write mode is `append` so Iceberg snapshots accumulate (no createOrReplace)


In [ ]:
with stage("silver-transform"):
    df = df_bronze
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))

    # currency: strip thousands separators before decimal cast
    money_cols = ["imposed_fine", "admin_costs"]
    for mc in money_cols:
        if mc in df.columns:
            df = df.withColumn(
                mc,
                F.regexp_replace(F.col(mc).cast("string"), ",", "").cast(DecimalType(10, 2)),
            )

    # explicit date parsing — inferSchema leaves these as strings
    for dc in ("violation_date", "last_modified_date"):
        if dc in df.columns:
            df = df.withColumn(dc, F.to_timestamp(dc, DATE_FORMAT_LONG))
    if "hearing_date" in df.columns:
        df = df.withColumn(
            "hearing_date",
            F.coalesce(
                F.to_timestamp("hearing_date", DATE_FORMAT_LONG),
                F.to_timestamp("hearing_date", DATE_FORMAT_SHORT),
            ),
        )

    for dc in ("latitude", "longitude"):
        if dc in df.columns:
            df = df.withColumn(dc, F.col(dc).cast(DoubleType()))
    for ic in ("ward", "street_number"):
        if ic in df.columns:
            df = df.withColumn(ic, F.col(ic).cast(IntegerType()))

    df = (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_path", F.lit(SOURCE_CSV_PATH))
    )
    df_silver = df
    df_silver.printSchema()
    df_silver.show(5, truncate=False)


In [ ]:
with stage("silver-write"):
    writer = (
        df_silver.writeTo(SILVER_TABLE)
                 .using("iceberg")
                 .partitionedBy(PARTITION_COLUMN)
                 .tableProperty("format-version", "2")
                 .tableProperty("write.parquet.compression-codec", "snappy")
                 .tableProperty("write.metadata.compression-codec", "gzip")
    )
    if not spark.catalog.tableExists(SILVER_TABLE):
        writer.create()
    else:
        writer.append()

    silver_count = spark.table(SILVER_TABLE).count()
    print(f"silver rows: {silver_count:,}")

    # Iceberg metadata tables (NOT SHOW PARTITIONS — that's not supported on Iceberg)
    spark.sql(f"SELECT * FROM {SILVER_TABLE}.snapshots ORDER BY committed_at DESC").show(5, truncate=False)
    spark.sql(f"SELECT partition, record_count FROM {SILVER_TABLE}.partitions").show(20, truncate=False)


## Deequ validation

Errors-level checks: completeness on the four critical columns, uniqueness
on `id`, non-negative on the currency columns, range checks on
`latitude` / `longitude`. The job halts the pipeline if any check fails,
and writes the failing rows to a quarantine Iceberg table.


In [ ]:
with stage("deequ-validate"):
    df_to_check = spark.table(SILVER_TABLE)

    check = (
        Check(spark, CheckLevel.Error, "chicago-violations-quality")
             .isComplete("id")
             .isComplete("docket_number")
             .isComplete("address")
             .isComplete("violation_date")
             .isUnique("id")
             .isNonNegative("imposed_fine")
             .isNonNegative("admin_costs")
             .isContainedIn("latitude", -90.0, 90.0)
             .isContainedIn("longitude", -180.0, 180.0)
    )

    result = (
        VerificationSuite(spark)
            .onData(df_to_check)
            .addCheck(check)
            .run()
    )
    result_df = VerificationResult.checkResultsAsDataFrame(spark, result)
    result_df.show(truncate=False)
    validation_passed = result.status == "Success"
    print(f"validation status: {result.status}")


### Quarantine — write rows that violate any predicate

Even when validation passes overall, individual records may have nulls
or out-of-range values; they are persisted to the quarantine table for
investigation.


In [ ]:
with stage("quarantine"):
    df_silver_full = spark.table(SILVER_TABLE)
    bad_predicate = (
        F.col("id").isNull()
        | F.col("docket_number").isNull()
        | F.col("address").isNull()
        | F.col("violation_date").isNull()
        | (F.col("imposed_fine") < 0)
        | (F.col("admin_costs") < 0)
        | (F.col("latitude") < -90) | (F.col("latitude") > 90)
        | (F.col("longitude") < -180) | (F.col("longitude") > 180)
    )
    df_quarantine = (
        df_silver_full.filter(bad_predicate)
                      .withColumn("_quarantined_at", F.current_timestamp())
    )
    quarantine_count = df_quarantine.count()
    print(f"quarantined rows: {quarantine_count:,}")

    if quarantine_count > 0:
        q_writer = (
            df_quarantine.writeTo(QUARANTINE_TABLE)
                         .using("iceberg")
                         .partitionedBy(PARTITION_COLUMN)
                         .tableProperty("format-version", "2")
        )
        if not spark.catalog.tableExists(QUARANTINE_TABLE):
            q_writer.create()
        else:
            q_writer.append()


## Gold stage — publish only validated rows

In [ ]:
with stage("gold-write"):
    if not validation_passed:
        raise RuntimeError("Deequ validation failed — refusing to write to gold. "
                           "See validation result_df above and the quarantine table.")

    df_gold = spark.table(SILVER_TABLE).filter(~bad_predicate)
    g_writer = (
        df_gold.writeTo(GOLD_TABLE)
               .using("iceberg")
               .partitionedBy(PARTITION_COLUMN)
               .tableProperty("format-version", "2")
               .tableProperty("write.parquet.compression-codec", "snappy")
    )
    if not spark.catalog.tableExists(GOLD_TABLE):
        g_writer.create()
    else:
        g_writer.append()

    gold_count = spark.table(GOLD_TABLE).count()
    print(f"gold rows: {gold_count:,}")
    spark.sql(f"SELECT * FROM {GOLD_TABLE}.snapshots ORDER BY committed_at DESC").show(5, truncate=False)


## Summary

In [ ]:
print("row counts:")
print(f"  bronze:     {bronze_count:,}")
print(f"  silver:     {silver_count:,}")
print(f"  quarantine: {quarantine_count:,}")
print(f"  gold:       {gold_count:,}")
print("\nstage timings (seconds):")
for name, secs in stage_times.items():
    print(f"  {name:20s} {secs:6.2f}")


## Querying the Iceberg tables

```sql
-- Latest data
SELECT * FROM chicago_lakehouse.violations.violations_gold WHERE ward = 14 LIMIT 10;

-- Snapshot history
SELECT * FROM chicago_lakehouse.violations.violations_gold.snapshots;

-- Partition stats
SELECT partition, record_count FROM chicago_lakehouse.violations.violations_gold.partitions;

-- Time travel — read an earlier snapshot
SELECT * FROM chicago_lakehouse.violations.violations_gold VERSION AS OF <snapshot_id>;
```
